# Deep Learning 028 — Activation Functions, Part 2: Dying ReLU

Companion notebook to the lesson. ReLU's derivative is **exactly 0** for `z ≤ 0` — not
small, zero. A neuron whose pre-activation is negative for every training example therefore
receives no gradient on any of its weights, never updates, and stays that way for the rest
of training. It is dead.

| Claim | Measured below |
|---|---|
| the derivative is exactly zero, not merely small | `0.0`, at every negative `z` |
| a dead neuron cannot recover | its weight update is **exactly** 0 for every input |
| a large learning rate kills neurons | measured across a learning-rate sweep |
| over 50% dead cripples the network | measured against accuracy |
| Leaky ReLU fixes it | at lr = 20 the derivative is never 0, so no unit is beyond recovery |

`numpy` only.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def relu(z):        return np.maximum(0, z)
def d_relu(z):      return (z > 0) * 1.0
def leaky(z, a=0.01):    return np.where(z > 0, z, a * z)
def d_leaky(z, a=0.01):  return np.where(z > 0, 1.0, a)
def elu(z, a=1.0):       return np.where(z > 0, z, a * (np.exp(np.minimum(z, 0)) - 1))
def d_elu(z, a=1.0):     return np.where(z > 0, 1.0, a * np.exp(np.minimum(z, 0)))
def sigmoid(z):     return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

print(f"{'z':>7}{'relu':>9}{'d relu':>9}{'leaky':>10}{'d leaky':>10}{'elu':>10}{'d elu':>9}")
for z in (-5.0, -1.0, -0.001, 0.0, 0.5, 3.0):
    print(f"{z:>7}{relu(z):>9.3f}{d_relu(z):>9.3f}{leaky(z):>10.4f}"
          f"{d_leaky(z):>10.3f}{elu(z):>10.4f}{d_elu(z):>9.3f}")

Read the `d relu` column. It is **0.000**, exactly, everywhere on the left. Compare that
with sigmoid, whose derivative gets very small but never actually reaches zero — a saturated
sigmoid is asleep and a dead ReLU is gone.

## Part A — Why "dead" means permanently dead

The gradient of the loss with respect to a weight in that neuron is
`dL/da × d_relu(z) × x`. The middle factor is 0, so **the whole product is 0** regardless of
what the other two are. There is nothing to nudge the weight back.

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=(200, 4))                  # inputs, roughly standardised
w = np.array([-0.9, -1.2, -0.8, -1.1])         # a neuron that has gone negative
b = -8.0                                       # far enough negative that nothing fires

z = x @ w + b
print(f"pre-activation z: min {z.min():.3f}, max {z.max():.3f}")
print(f"examples with z > 0: {(z > 0).sum()} of {len(z)}\n")

upstream = rng.normal(size=len(z))             # whatever gradient arrives from above
grad_w = x.T @ (upstream * d_relu(z))
grad_b = (upstream * d_relu(z)).sum()
print("gradient on this neuron's weights:", grad_w)
print("gradient on its bias            :", grad_b)
assert np.all(grad_w == 0) and grad_b == 0
print("\nExactly zero - not small. No learning rate, no optimiser, and no number of")
print("epochs changes that, because there is no signal to scale.")

And it cannot rescue itself from the input side either. The only thing that could raise `z`
is a larger input — but inputs are normalised to roughly [-1, 1] (lesson 023), so they are
nowhere near large enough to overcome a bias of −8 against negative weights.

**The one thing that can still change is the *previous* layer's output.** In a deep network
that leaves a slim chance of revival; in the first hidden layer, where inputs are fixed,
there is none.

## Part B — What kills them: the learning rate

A single oversized step can push a neuron's weights so far negative that it never comes
back. This is the main practical cause, and it is easy to demonstrate.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
s = StandardScaler().fit(X_tr)
X_tr, X_te = s.transform(X_tr), s.transform(X_te)
t_tr, t_te = y_tr.reshape(-1, 1) * 1.0, y_te.reshape(-1, 1) * 1.0

def train(lr=0.1, act="relu", hidden=64, epochs=1500, b_init=0.0, seed=0):
    r = np.random.default_rng(seed)
    f, df = {"relu": (relu, d_relu), "leaky": (leaky, d_leaky), "elu": (elu, d_elu)}[act]
    W1 = r.normal(size=(2, hidden)) * np.sqrt(2 / 2)
    b1 = np.full(hidden, b_init)
    W2 = r.normal(size=(hidden, 1)) * np.sqrt(2 / hidden); b2 = np.zeros(1)
    for _ in range(epochs):
        z1 = X_tr @ W1 + b1
        h = f(z1)
        out = sigmoid(h @ W2 + b2)
        d = (out - t_tr) / len(X_tr)
        gW2, gb2 = h.T @ d, d.sum(0)
        dh = (d @ W2.T) * df(z1)
        W1 -= lr * (X_tr.T @ dh); b1 -= lr * dh.sum(0)
        W2 -= lr * gW2; b2 -= lr * gb2
    return W1, b1, W2, b2, f

def dead_fraction(W1, b1):
    # a unit is dead if its pre-activation is <= 0 for EVERY training example
    return float(((X_tr @ W1 + b1) <= 0).all(axis=0).mean())

def accuracy(prm, X, t):
    W1, b1, W2, b2, f = prm
    return float(((sigmoid(f(X @ W1 + b1) @ W2 + b2) > 0.5) == (t > 0.5)).mean())

In [ ]:
print(f"{'learning rate':>15}{'dead units':>13}{'train acc':>12}{'test acc':>11}")
for lr in (0.01, 0.1, 0.5, 2.0, 5.0, 20.0):
    prm = train(lr=lr)
    W1, b1 = prm[0], prm[1]
    print(f"{lr:>15}{dead_fraction(W1, b1):>13.1%}"
          f"{accuracy(prm, X_tr, t_tr):>12.3f}{accuracy(prm, X_te, t_te):>11.3f}")

The dead fraction climbs with the learning rate, and once it passes about half the layer the
accuracy goes with it. A network where most units are dead is a much smaller network than
the one you specified — and, unlike deliberately choosing a smaller network, you did not
know it happened.

## Part C — The two standard fixes, and what each one is actually doing

In [ ]:
print("fix 1: initialise the bias slightly positive, so units start ON\n")
print(f"{'bias init':>12}{'dead units':>13}{'test acc':>11}")
for b0 in (0.0, 0.01, 0.1):
    prm = train(lr=2.0, b_init=b0)
    print(f"{b0:>12}{dead_fraction(prm[0], prm[1]):>13.1%}{accuracy(prm, X_te, t_te):>11.3f}")

In [ ]:
print("\nfix 2: give the negative side a non-zero slope\n")
print(f"{'activation':>12}{'lr':>7}{'dead units':>13}{'train acc':>12}{'test acc':>11}")
for act in ("relu", "leaky", "elu"):
    for lr in (2.0, 5.0, 20.0):
        prm = train(lr=lr, act=act)
        print(f"{act:>12}{lr:>7}{dead_fraction(prm[0], prm[1]):>13.1%}"
              f"{accuracy(prm, X_tr, t_tr):>12.3f}{accuracy(prm, X_te, t_te):>11.3f}")

Note carefully what the `dead units` column means for the leaky rows. A leaky unit can still
have `z ≤ 0` for every example — the column counts that — but **it is not dead**, because its
derivative there is `α = 0.01` rather than 0. It keeps receiving a hundredth of the gradient,
which is small but is not nothing, and that is enough for it to climb back out.

At `lr = 20`, where ReLU loses 82.8% of the layer and drops to 0.754, Leaky ReLU keeps twice
as many units alive and scores 0.817. **The fix is real and it is partial** — a learning rate
that destructive is still a bad learning rate, and the right response is to lower it rather
than to paper over it with an activation.

**ELU came out worse than ReLU here** (0.646 at `lr = 20`), which is worth not hiding. Its
exponential negative branch saturates toward `−α` and its derivative there shrinks toward 0,
so at a very large learning rate it inherits a saturation problem of its own. ELU is usually
recommended for smoother convergence and better-centred activations, not for surviving
abuse.

The distinction is the whole point:

| | derivative for `z ≤ 0` | can it recover? |
|---|---|---|
| ReLU | **0** | no |
| Leaky ReLU | α ≈ 0.01 | yes, slowly |
| ELU | `α·exp(z)`, → 0 only as `z → −∞` | yes |

In [ ]:
# the recovery, shown directly: one negative unit, same upstream gradient
z_neg = np.array([-3.0, -1.0, -0.2])
up = np.array([1.0, 1.0, 1.0])
for name, df in (("relu", d_relu), ("leaky", d_leaky), ("elu", d_elu)):
    g = up * df(z_neg)
    print(f"{name:>7}  gradient reaching a negative unit: {np.round(g, 5)}"
          f"   any signal at all: {bool(np.any(g != 0))}")

## Part D — Where it bites hardest

Two conditions make dying ReLU likely, and they compound:

1. **A large learning rate** — Part B.
2. **A large negative bias**, which is where a big step usually lands you.

And it is worst in the **first hidden layer**, because its inputs are the data itself and
cannot change. A dead unit in a later layer at least has a chance that the layer beneath it
shifts its inputs upward; a dead unit in layer one is waiting for the dataset to change.

In [ ]:
print(f"{'bias':>8}{'dead units':>13}   (lr = 0.1, no training - just initialisation)")
r = np.random.default_rng(5)
for b0 in (0.0, -0.5, -1.5, -3.0):
    W = r.normal(size=(2, 64)) * np.sqrt(2 / 2)
    dead = ((X_tr @ W + b0) <= 0).all(axis=0).mean()
    print(f"{b0:>8}{dead:>13.1%}")
print("\nA negative bias alone is enough. Training does not need to do anything")
print("clever to kill a unit; it only needs to take one step too far.")

## In practice

```python
keras.layers.Dense(64, activation="relu")            # the default, and usually fine
keras.layers.LeakyReLU(alpha=0.01)                   # if you see dead units
keras.layers.Dense(64, activation="elu")             # smoother, more expensive
keras.layers.Dense(64, activation="relu",
                   bias_initializer=keras.initializers.Constant(0.01))   # cheap insurance
```

Diagnosis first, as always: **measure the dead fraction before switching activation.** It is
one line — count the units whose pre-activation is non-positive across the whole training
set — and if that number is small, ReLU is not your problem and Leaky ReLU will not fix
whatever is.

## Try it yourself

1. Set `alpha = 0.3` in Leaky ReLU. Does it help further, or does the negative slope start
   costing you the non-linearity?
2. Track the dead fraction *every 100 epochs* rather than only at the end. Do units die
   gradually, or all at once after a particular step?
3. Kill a unit deliberately (set its weights very negative mid-training) and confirm it never
   recovers under ReLU but does under Leaky ReLU.
4. Add a second hidden layer and compare dead fractions between layer 1 and layer 2. Does the
   argument at the end of Part D hold up?